# Exercise 1. Word Embeddings 101
This exercise doesn't have an RQ, but is all about exploring how (static) word embeddings work!

## 1.1 Load a Pretrained Embedding Model
Start by installing the package `gensim`: 

In [2]:
%pip install gensim


[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


Let's import `pathlib` and `KeyedVectors` from `gensim.models` 

In [3]:
from pathlib import Path
from gensim.models import KeyedVectors

Load the `pre-installed` model `GloVe`

In [4]:
path = Path.cwd()
models_path = path.parents[1] / "resources" / "models" 

# note the str() as KeyedVectors.load() needs a string path !
model = KeyedVectors.load(str(models_path / "glove_wiki_gigaword_300.kv")) 

:::{admonition} What is GloVe? Can I use other models?
:class: tip, dropdown
`GloVe` word vectors are trained on aggregated global word-word co-occurrence statistics from a corpus. The `glove-wiki-gigaword-300` vectors are 300-dimensional and trained on co-occurence statistics from Wikipedia articles and various news outlets. See also {cite:t}`pennington_glove_2014`

You can download (other) models such as `Word2Vec` as such: 
```python
import gensim.downloader as api
model = api.load("word2vec-google-news-300") 
```
See [Models](https://github.com/piskvorky/gensim-data?tab=readme-ov-file#models) for the model overview and read more about [gensim.downloader](https://radimrehurek.com/gensim/downloader.html).
:::


## 1.2 Exploring Polysemous Words
Polysemy refers to words that have two or more meanings. There are many out there:
```{figure} ../figures/class3/polysemy.png
---
name: polysemy
width: 80%
---
```

We can look at the most similar words (cosine similarity) for the word `leaves` in our model to illustrate this:

In [5]:
model.most_similar("leaves", topn=15)

[('leaf', 0.5564894080162048),
 ('stems', 0.554435670375824),
 ('leaving', 0.5414506196975708),
 ('leave', 0.5339293479919434),
 ('ends', 0.526331901550293),
 ('flowers', 0.48853448033332825),
 ('stalks', 0.47854116559028625),
 ('grows', 0.47765129804611206),
 ('arrives', 0.47416287660598755),
 ('takes', 0.47301051020622253),
 ('foliage', 0.4626350700855255),
 ('turns', 0.4491370916366577),
 ('goes', 0.44434431195259094),
 ('becomes', 0.43927299976348877),
 ('comes', 0.43823492527008057)]

:::{admonition} Question
:class: red
Take a look at the words above - what do you notice? Speak with a friend about it!
:::


### Your Turn: Try it out!
:::{admonition} HANDS-ON
:class: red
As a simple starting exercise, try to find a polysemous word like `leaves` or `bat` where the top-15 most similar words (according to cosine similarity) contains related words from several meanings.
- Use `model.most_similar()` for this and remember to write words in *lowercase*

If you are blanking on words, you can use Google to get inspiration, but once you begin looking for them (on the internet or in your mind), you'll notice how many there are!
:::


## 1.3 Synonyms and Antonyms

Let's explore synonyms and antonyms with word embeddings. One approach is *cosine distance*, defined as `1 - cosine similarity`. 

The higher the distance, the farther apart two words are in vector space. This means that they should have less "in common", but word embeddings are not always intuitive!

For example, an antonym for the word `"happy"` is the word `"sad"`:

In [ ]:
model.distance("happy", "sad")

0.43471431732177734

And a synonym for `happy` would be`cheerful`:

In [ ]:
model.distance("happy","cheerful") # remember, the higher the distance, the less similar!

0.5596834421157837

Although `happy` and `cheerful` are synonyms, `happy` is actually closer to `sad` than to `cheerful` in terms of cosine distance! We can also check this programatically with the "less than" boolean operator `<`

In [29]:
# if true, happy is closer to sad than to cheerful
model.distance("happy", "sad") < model.distance("happy","cheerful") 

True

:::{admonition} QUESTION
:class: red
With your knowledge about *embeddings*, can you explain why *happy* is closer to *sad* than *cheerful*?
:::

### Your Turn: Finding Counter-Intutive Synonym-Antonym pairs!
:::{admonition} HANDS-ON & QUESTION
:class: red

**TASK 1**

Find three words (`w1`,`w2`,`w3`) where `w1` and `w2` are synonyms and `w1` and `w3` are antonyms, but where `w1` is closer to its antonym `w3` than to its synonym `w2`:

`Cosine Distance(w1,w3) < Cosine Distance(w1,w2)`

This is just what we did above!

**TASK 2**

When you have found your example, try to see if you can explain exactly that *counter-intuitive* example happened:
- Write a few bullets in your notebook!

:::


## 1.4 Exploring Word Analogies
An interesting feature of trained word embeddings is that we can discover *analogies* by doing basic arithmetic (addition/subtraction). That is, the embedding for *queen* can be approximated as:

$$
\mathbf{w}_{\text{queen}} \approx \mathbf{w}_{\text{king}} - \mathbf{w}_{\text{man}} + \mathbf{w}_{\text{woman}}"
$$

Corresponding to the statement "**man** is to **king** as **woman** is to **queen**. Highly recommend reading {cite:t}`allen_analogies_2019` for deeper insights here!


In Python, we can use the method `most_similar()` on our model as such (bit weird syntax!):

In [21]:
model.most_similar(positive=['king', 'woman'], 
                   negative=['man'])[0]

('queen', 0.7118192911148071)

> NB: Depending on the model, it might not be exactly the vector for queen, but could be e.g., "princess" or "monarch" :)

### Your Turn: "Correct" & "Wrong" Analogies
Please do this task with a classmate or two! Experiment and discuss!

:::{admonition} HANDS-ON
:class: red

Use the `.most_similar()` function on your model and do the following tasks:

**TASK A**  
Find "correct" analogies. By "correct," I mean analogies where the closest vector corresponds to the word you expect (like the king–queen example). You should:  
1. Try to find *at least two* analogies.

**TASK B**  
Find "wrong" analogies. By "wrong," I mean analogies that *should* work (like the king–queen example) but *do not*. You should:  
1. Try to find *at least one* where the closest vector does not correspond to what you expect!
2. Discuss with your classmate why you think your analogies do not work.
:::

My solution

In [ ]:
# analogies that hold! 
print(["[INFO:] Analogies that hold!"])
print(model.most_similar(positive=["god", "woman"], 
                    negative=["man"])[0]) # god - man + woman = goddess

print(model.most_similar(positive=["paris", "italy"],
                        negative=["france"])[0]) # paris - france + italy = rome

print(model.most_similar(positive=["psychology", "humanities"],
                          negative=["statistics"])[0]) # psychology - statistics + humanities = anthropology

print(model.most_similar(positive=["comedy", "scary"], negative=["funny"])[0]) # comedy - funny + scary = horror

# print space between the two sections
print("\n")

# analogy that does not hold
print("[INFO:] Analogies that should hold, but do not!")
print(model.most_similar(positive=["ship", "road"], negative=["ocean"])[0]) # ship - ocean + road = "roads" (does also not work if you replace road with wheels)
print(model.most_similar(positive=["trousers", "hot"], negative=["long"])[0]) # trousers - long + hot = "shorts" (does not work?)

['[INFO:] Analogies that hold!']
('goddess', 0.5733169317245483)
('rome', 0.7368948459625244)
('anthropology', 0.6434165835380554)
('horror', 0.5571268796920776)


[INFO:] Analogies that should hold, but do not!
('roads', 0.45349568128585815)
('pants', 0.57420814037323)


## 1.5 Uncovering Bias
Due to the way embeddings are trained, they may contain some biases. While basic heuristics are fine, gender stereotypes can be problematic when applying these embeddings in the real world!

Let's try: 
`director - man + woman`

In [117]:
print(model.most_similar(positive=["director", "woman"],
                          negative=['man']))

[('assistant', 0.4793030619621277), ('chairwoman', 0.4766245186328888), ('executive', 0.46431592106819153), ('susan', 0.4622037410736084), ('spokeswoman', 0.45612895488739014), ('associate', 0.4546034634113312), ('directors', 0.44585469365119934), ('actress', 0.4415530860424042), ('deputy', 0.42916229367256165), ('researcher', 0.41419851779937744)]


## 1.6 Visualization